# Hierarchical English summarization

Summarize long English text with `facebook/bart-large-cnn`. The model is downloaded on first use and can be slow on a CPU. Install Transformers, PyTorch, and a supported tokenizer.

Chunks are measured using the model tokenizer, including for a single unusually long sentence. Long intermediate summaries are reduced in multiple rounds. The outputs below were generated with the current chunking and summarization code.

**Data:** No external dataset. The input texts are embedded in the notebook; model weights are downloaded separately.

In [1]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device).eval()

def hierarchical_summarize(text: str, chunk_max_tokens: int = 800,
                          intermediate_min_length: int = 50,
                          intermediate_max_length: int = 150,
                          final_min_length: int = 100,
                          final_max_length: int = 300) -> str:
    """Summarize tokenizer-sized chunks until one summary remains."""
    if not text.strip():
        raise ValueError("Provide a non-empty text to summarize.")
    model_limit = tokenizer.model_max_length - tokenizer.num_special_tokens_to_add(pair=False)
    if not 0 < chunk_max_tokens <= model_limit:
        raise ValueError(f"chunk_max_tokens must be between 1 and {model_limit}.")
    current = text.strip()
    for _ in range(10):
        tokens = tokenizer.encode(current, add_special_tokens=False)
        chunks = [tokens[i:i + chunk_max_tokens] for i in range(0, len(tokens), chunk_max_tokens)]
        summaries = []
        for chunk in chunks:
            chunk_text = tokenizer.decode(chunk, skip_special_tokens=True)
            single = len(chunks) == 1
            max_len = final_max_length if single else intermediate_max_length
            min_len = final_min_length if single else intermediate_min_length
            # Short inputs should not be forced to produce a longer summary.
            max_len = max(2, min(max_len, len(chunk)))
            min_len = max(0, min(min_len, max_len - 1, len(chunk) // 4))
            inputs = tokenizer(chunk_text, return_tensors="pt", truncation=True).to(device)
            with torch.inference_mode():
                output = model.generate(**inputs, min_length=min_len, max_length=max_len,
                                        length_penalty=2.0, num_beams=4, do_sample=False)
            summaries.append(tokenizer.decode(output[0], skip_special_tokens=True))
        if len(summaries) == 1:
            return summaries[0]
        combined = " ".join(summaries)
        if len(tokenizer.encode(combined, add_special_tokens=False)) >= len(tokens):
            raise RuntimeError("Summaries did not get shorter; reduce output length or increase chunk size.")
        current = combined
    raise RuntimeError("Could not reduce the text to a single chunk in 10 rounds.")


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [2]:
long_english_text = """Lithium-ion batteries are found in everything from smartphones to cars, and while they are generally very safe if stored and charged correctly, there are thousands of documented cases of them catching fire — sometimes with deadly consequences.

Lithium-ion batteries contain flammable electrolytes — liquid solutions of lithium salts dissolved in organic solvents that allow the electric charge to flow. The batteries can become unstable under certain conditions, such as physical damage like piercing, overcharging, extreme temperatures or manufacturing defects. When things go wrong, a battery can heat up and catch fire very rapidly, undergoing a dangerous chain reaction called a “thermal runaway.”

Commercial aviation is especially exposed to the problem, because of how ubiquitous battery-powered gadgets are on planes and how dangerous a fire in the cabin or cargo hold can be. In the US, the Federal Aviation Administration (FAA) has long prohibited carrying spare lithium-ion batteries in checked baggage and mandates that all batteries brought into the cabin must remain accessible. The agency recorded 89 battery incidents involving smoke, fire or extreme heat on passenger and cargo aircraft in 2024, and 38 in the first half of 2025.

These incidents can lead to the total loss of an aircraft, like the Airbus A321 that was gutted by flames in January in Busan, South Korea. The fire was likely started by a power bank stored in an overhead compartment, according to investigators, which has led some airlines to ban the devices.

But the risks of thermal runaways extend to homes, particularly vulnerable to e-bike or e-scooter battery fires, and businesses of all kinds: a survey conducted by insurance provider Aviva in 2024, among over 500 UK businesses found that just over half had experienced an incident linked to lithium-ion batteries, such as sparking, fires and explosions.

Researchers all over the world are working to solve the problem by developing safer batteries, for example by replacing the liquid electrolyte with a more fire-resistant solid or gel. However, such solutions require significant changes to current production lines, an obstacle to widespread adoption.

Now, a team of researchers from The Chinese University of Hong Kong has proposed a change in lithium-ion battery design that could rapidly integrate into current manufacturing methods, because it simply involves swapping chemicals in the existing electrolyte solution.

The method was detailed earlier this year in a study led by Yue Sun, now a postdoctoral fellow at Virginia Tech: “I think the most difficult thing for people to realize about batteries is that when you try to optimize performance, sometimes you compromise safety,” she said, explaining that increasing performance requires a focus on chemical reactions that happen at room temperature, while increasing safety focuses on reactions that happen at high temperatures.

“So we came up with an idea to break this trade off by designing a temperature-sensitive material, which can provide a good performance at room temperature, but can also offer good stability at high temperatures.”

Staying cool

Battery fires generally start when part of the electrolyte breaks down under stress and releases heat in a chain reaction. Sun and her colleagues’ design uses a new electrolyte that includes two solvents to stop this chain reaction in its tracks.

At room temperature, the first solvent keeps the battery’s chemical structure tight, optimizing performance, but if the battery starts to heat up, the second solvent takes over and prevents a fire by loosening that structure and slowing down reactions that could result in a thermal runaway.

In lab tests, a battery with this new design that was pierced with a nail saw its temperature rise by only 3.5°C (38.3° Fahrenheit), instead of the 555°C (1,031° Fahrenheit) spike that occurred with a traditional battery. The researchers say there is no negative impact on the battery’s performance or durability, and it retained over 80% of its capacity after 1,000 charging cycles.

“Since our invention is the electrolyte, it can be easily implemented into commercially available systems — you basically just swap in the new electrolyte,” said Yi-Chun Lu, a professor of mechanical and automation engineering at The Chinese University of Hong Kong, and one of the authors of the study.

“In the manufacturing process, the most difficult aspect is the electrodes, or the solid part. But the electrolyte that we are swapping is a liquid, so you can directly inject it into the cell without any new equipment or new process,” Lu added.

The new chemical recipe would slightly increase the manufacturing cost, but Lu said that at scale, the price would be “at a very similar level” to current batteries. The researchers are interested in pursuing the idea commercially, and are currently in discussion with battery manufacturers to bring the design to market, which could take between three to five years, according to Lu.

In tests, the researchers made a battery large enough to power a tablet, but Lu said that there needs to be more “validation” to scale the design up to the size required for cars, for example.

Researchers in the field of lithium battery safety who didn’t participate in the study expressed positive views of the work when contacted by CNN for comment. Donal Finegan, a senior scientist at the US National Renewable Energy Laboratory, said in an email that the new design is an exciting development, and it translates into a safety battery that can tolerate hot conditions and short circuiting without causing a fire: “The strategically chosen electrolyte solution is scalable and does not significantly impede the cycle life of the battery, which helps eliminate many barriers towards mass adoption in commercial battery systems,” he added.

The study explored a range of electrolyte compositions and found options where there was a balance between battery cells lasting many cycles while also maintaining stability at higher temperatures, according to Gary Koenig, a professor of chemical engineering at the University of Virginia. “From a manufacturing perspective, implementing a new electrolyte can be achieved in a relatively short timeframe as long as there is not an unforeseen processing compatibility issue,” he said in an email.

Jorge Seminario, a professor of chemical engineering at Texas A&M University, said in an email that the new design addresses one of the most critical challenges in high-energy lithium-ion batteries, which is achieving both safety and performance: “The study is highly innovative and impactful, offering a practical solution to a critical bottleneck in lithium-ion battery safety,” he added."""

result = hierarchical_summarize(long_english_text)
print(result)

Lithium-ion batteries can become unstable under certain conditions. A new design uses a new electrolyte that includes two solvents to stop this chain reaction. The new design can be easily implemented into commercially available systems.


In [3]:
long_english_text = """A Russian general has been killed in a car bombing in Moscow, with officials pointing the finger at Ukraine for the latest apparent assassination of a senior military officer.

Lt. Gen. Fanil Sarvarov, who ran the armed forces operational training department, died after an explosive device installed under the chassis of a car exploded on Monday morning, Russia’s investigative committee said in a statement.

“Investigators are pursuing various motives for the murder. One of the theories is that the crime was organized by Ukrainian special services,” the committee added.

Russian President Vladimir Putin was “immediately informed” of the death of Sarvarov via special services channels, according to the Kremlin.

Several prominent Russians have been killed by explosive devices or shot dead in Moscow in attacks blamed on the Ukrainian security services since the full-scale Russian invasion of Ukraine in 2022.

Video released by Russian state media Monday showed a badly damaged vehicle in a parking lot in a middle-class neighborhood of Moscow.

Sarvarov had been head of the training department for nine years, according to state news agency TASS. The 56-year-old had previously “carried out the tasks of organizing and conducting an operation in Syria,” when Russian forces were backing the Assad regime, TASS said.

Other recent deaths

Senior Russian officers killed in Moscow have included Lt. Gen. Yaroslav Moskalik, deputy head of the main operational department of the General Staff, who was the victim of a car bomb attack near Moscow in April.

A year ago, Igor Kirillov, a general in charge of the Russian military’s nuclear and chemical weapons protection forces, died after an explosive device planted on a scooter was detonated near the entrance to a residential building. Ukraine swiftly claimed responsibility for that assasination.

Others to have been killed include the founder of a pro-Russian militia group, Armen Sarkisyan, described by Ukraine as a “criminal mastermind,” who died following a bombing in central Moscow, in February this year.

Several prominent political figures have also died, including a leading figure in Russia’s nationalist movement, Darya Dugina, in 2022. An influential pro-war blogger, Vladlen Tatarsky, was killed in a cafe bombing in St Petersburg in April 2023."""

result = hierarchical_summarize(long_english_text)
print(result)

Lt. Gen. Fanil Sarvarov died in a car bombing in a middle-class neighborhood of Moscow. The 56-year-old had been head of the training department for nine years. Russia's investigative committee says one theory is that the crime was organized by Ukrainian special services. Several prominent Russians have been killed by explosive devices or shot dead in Moscow in attacks blamed on the Ukrainian security services since the full-scale Russian invasion of Ukraine in 2022. Others to have died include the founder of a pro-Russian militia group, Armen Sarkisyan, who died in February.
